In [2]:
%matplotlib inline

In [3]:
import crested
import numpy as np
import pandas as pd
import anndata as ad

2025-03-21 14:04:09.517985: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742565849.532497  295635 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742565849.537171  295635 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1742565849.550960  295635 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1742565849.550974  295635 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1742565849.550976  295635 computation_placer.cc:177] computation placer alr

In [4]:
import tensorflow as tf

# Check GPU availability
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

# Check TensorFlow build info
print(tf.sysconfig.get_build_info())

# Check CUDA version
print(tf.test.is_built_with_cuda())

# Check cuDNN version
print(tf.test.is_built_with_gpu_support())

Num GPUs Available: 1
OrderedDict([('cpu_compiler', '/usr/lib/llvm-18/bin/clang'), ('cuda_compute_capabilities', ['sm_60', 'sm_70', 'sm_80', 'sm_89', 'compute_90']), ('cuda_version', '12.5.1'), ('cudnn_version', '9'), ('is_cuda_build', True), ('is_rocm_build', False), ('is_tensorrt_build', False)])
True
True


In [5]:
data_dir = '/rds/project/rds-SDzz0CATGms/users/bt392/15_Kat_Atlas/results/data/'

In [6]:
import scipy.io

# Load the sparse matrix from the Matrix Market format
matrix = scipy.io.mmread(data_dir + "CITE_eRNA_pseudobulked.mtx")

In [7]:
matrix

<COOrdinate sparse matrix of dtype 'float64'
	with 1359474 stored elements and shape (73316, 49)>

In [8]:
adata = ad.AnnData(matrix.T.toarray())


In [9]:
meta = pd.read_csv(data_dir + "meta_pseudobulk.csv")

In [10]:
meta.index = meta['celltype']

In [11]:
peaks = pd.read_csv(data_dir + "peaks_filted.csv")

In [12]:
peaks.index = peaks['peak']

In [13]:
peaks = peaks[['chr', 'start', 'end']]

In [14]:
adata.obs = meta

In [15]:
adata.var = peaks

In [16]:
adata

AnnData object with n_obs × n_vars = 49 × 73316
    obs: 'celltype'
    var: 'chr', 'start', 'end'

In [17]:
crested.pp.train_val_test_split(
    adata,
    strategy="region",
    val_size=0.1,
    test_size=0.1,
    shuffle=True,
    random_state=42,
)

In [18]:
print(adata.var["split"].value_counts())
adata.var

split
train    58652
val       7332
test      7332
Name: count, dtype: int64


,chr,start,end,split
peak,,,,
chr1:4699342-4699722,chr1,4699342,4699722,train
chr1:4705129-4705433,chr1,4705129,4705433,train
chr1:4714121-4714574,chr1,4714121,4714574,train
chr1:4729397-4729734,chr1,4729397,4729734,train
chr1:4731628-4731950,chr1,4731628,4731950,val
...,...,...,...,...
chrY:90797302-90798030,chrY,90797302,90798030,train
chrY:90798311-90799235,chrY,90798311,90799235,train
chrY:90799252-90799852,chrY,90799252,90799852,train


In [32]:
crested.pp.change_regions_width(
    adata, 2114
)  # change the adata width of the regions to 2114bp

2025-03-21T13:41:52.235684+0000 WARNING Chromsizes file not provided. Will not check if regions are within chromosomes


In [33]:
#crested.pp.normalize_peaks(adata)

In [35]:
# %matplotlib inline
# crested.pl.bar.normalization_weights(adata, title="Normalization Weights per Cell Type")

In [36]:
# This one is a bit extreme, so either something should change with the inputs or just no normalisation

In [37]:
# Set the genome
genome = crested.Genome(
    "/home/bt392/rds/rds-bg200-hphi-gottgens/references/10x/refdata-cellranger-arc-mm10-2020-A-2.0.0/fasta/genome.fa", 
    "/home/bt392/rds/rds-bg200-hphi-gottgens/users/bt392/10_Eomes_invitro_gut/results/atac/chrombpnet/mm10.chrom.sizes"
)
crested.register_genome(
    genome
)  # Register the genome so that it can be used by the package

2025-03-21T13:42:07.354133+0000 INFO Genome genome registered.


In [38]:
datamodule = crested.tl.data.AnnDataModule(
    adata,
    batch_size=256,  # lower this if you encounter OOM errors
    max_stochastic_shift=3,  # optional data augmentation to slightly reduce overfitting
    always_reverse_complement=True,  # default True. Will double the effective size of the training dataset.
)

In [39]:
crested.tl.zoo.chrombpnet

<function crested.tl.zoo._chrombpnet.chrombpnet(seq_len: int, num_classes: int, first_conv_filters: int = 512, first_conv_filter_size: int = 5, first_conv_pool_size: int = 0, first_conv_activation: str = 'gelu', first_conv_l2: float = 1e-05, first_conv_dropout: float = 0.1, n_dil_layers: int = 8, num_filters: int = 512, filter_size: int = 3, activation: str = 'relu', output_activation: str = 'softplus', l2: float = 1e-05, dropout: float = 0.1, batch_norm: bool = True, dense_bias: bool = True) -> keras.src.models.model.Model>

In [40]:
# Load chrombpnet architecture for a dataset with 2114bp regions 

# Their default model is slightly different from the original bpnet
# mainly the width of the first convolutional filter is way smaller
model_architecture = crested.tl.zoo.chrombpnet(seq_len=2114, num_classes=49)

I0000 00:00:1742564531.028364  292489 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79077 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:41:00.0, compute capability: 8.0


In [41]:
# Load the default configuration for training a topic classication model
from crested.tl import default_configs, TaskConfig

config = default_configs("peak_regression")
print(config)

# If you want to change some small parameters to an existing config, you can do it like this
# For example, the default learning rate is 0.001, but you can change it to 0.0001
# config.optimizer.learning_rate = 0.0001

TaskConfig(optimizer=<keras.src.optimizers.adam.Adam object at 0x15374c391710>, loss=<crested.tl.losses._cosinemse_log.CosineMSELogLoss object at 0x15374ddebcd0>, metrics=[<MeanAbsoluteError name=mean_absolute_error>, <MeanSquaredError name=mean_squared_error>, <CosineSimilarity name=cosine_similarity>, <PearsonCorrelation name=pearson_correlation>, <ConcordanceCorrelationCoefficient name=concordance_correlation_coefficient>, <PearsonCorrelationLog name=pearson_correlation_log>, <ZeroPenaltyMetric name=zero_penalty_metric>])


In [42]:
# setup the trainer
trainer = crested.tl.Crested(
    data=datamodule,
    model=model_architecture,
    config=config,
    project_name="/rds/project/rds-SDzz0CATGms/users/bt392/15_Kat_Atlas/results/crested",  # change to your liking
    run_name="trial1",  # change to your liking
    logger=None,  # or 'wandb', 'tensorboard'
)

2025-03-21T13:42:14.195037+0000 WARNING Output directory /rds/project/rds-SDzz0CATGms/users/bt392/15_Kat_Atlas/results/crested/trial1/checkpoints already exists. Will continue training from epoch 3.
2025-03-21T13:42:14.195441+0000 WARNING Loading a model with compile=True. The CREsted config object will be ignored.


In [ ]:
# train the model
trainer.fit(epochs=25)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence            │ (None, 2114, 4)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 2114, 512) │     10,240 │ sequence[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 2114, 512) │      2,048 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 2114, 512) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 2114, 512) │          0 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_1conv         │ (None, 2110, 512) │    786,432 │ dropout[0][0]     │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_1bn           │ (None, 2110, 512) │      2,048 │ bpnet_1conv[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_1activation   │ (None, 2110, 512) │          0 │ bpnet_1bn[0][0]   │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_1crop         │ (None, 2110, 512) │          0 │ dropout[0][0]     │
│ (Cropping1D)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 2110, 512) │          0 │ bpnet_1activatio… │
│                     │                   │            │ bpnet_1crop[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_1dropout      │ (None, 2110, 512) │          0 │ add[0][0]         │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_2conv         │ (None, 2102, 512) │    786,432 │ bpnet_1dropout[0… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_2bn           │ (None, 2102, 512) │      2,048 │ bpnet_2conv[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_2activation   │ (None, 2102, 512) │          0 │ bpnet_2bn[0][0]   │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_2crop         │ (None, 2102, 512) │          0 │ bpnet_1dropout[0… │
│ (Cropping1D)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 2102, 512) │          0 │ bpnet_2activatio… │
│                     │                   │            │ bpnet_2crop[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_2dropout      │ (None, 2102, 512) │          0 │ add_1[0][0]       │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 19,017,365 (72.55 MB)

 Trainable params: 6,336,049 (24.17 MB)

 Non-trainable params: 9,216 (36.00 KB)

...

458/459 ━━━━━━━━━━━━━━━━━━━━ 0s 438ms/step - concordance_correlation_coefficient: 0.0010 - cosine_similarity: 0.5494 - loss: 1.6042 - mean_absolute_error: 0.0529 - mean_squared_error: 63.2612 - pearson_correlation: 0.0031 - pearson_correlation_log: 0.0482 - zero_penalty_metric: 678.8425

2025-03-21 13:47:43.416956: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_7173', 192 bytes spill stores, 192 bytes spill loads

2025-03-21 13:47:43.431735: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_5816', 168 bytes spill stores, 168 bytes spill loads

2025-03-21 13:47:43.478691: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_5816', 24 bytes spill stores, 24 bytes spill loads

2025-03-21 13:47:43.516453: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_5816', 24 bytes spill stores, 24 bytes spill loads

2025-03-21 13:47:43.853789: I extern

459/459 ━━━━━━━━━━━━━━━━━━━━ 0s 519ms/step - concordance_correlation_coefficient: 0.0010 - cosine_similarity: 0.5494 - loss: 1.6038 - mean_absolute_error: 0.0529 - mean_squared_error: 63.2092 - pearson_correlation: 0.0031 - pearson_correlation_log: 0.0482 - zero_penalty_metric: 677.6118

2025-03-21 13:48:25.318048: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_684', 24 bytes spill stores, 24 bytes spill loads

2025-03-21 13:48:25.402964: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_684', 168 bytes spill stores, 168 bytes spill loads

2025-03-21 13:48:25.409392: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_684', 4 bytes spill stores, 4 bytes spill loads

2025-03-21 13:48:25.450502: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_684', 40 bytes spill stores, 40 bytes spill loads

2025-03-21 13:48:25.527119: I external/local

459/459 ━━━━━━━━━━━━━━━━━━━━ 392s 591ms/step - concordance_correlation_coefficient: 0.0010 - cosine_similarity: 0.5495 - loss: 1.6033 - mean_absolute_error: 0.0529 - mean_squared_error: 63.1575 - pearson_correlation: 0.0031 - pearson_correlation_log: 0.0481 - zero_penalty_metric: 676.3865 - val_concordance_correlation_coefficient: 4.0327e-09 - val_cosine_similarity: 0.5447 - val_loss: 1.3175 - val_mean_absolute_error: 0.0165 - val_mean_squared_error: 0.6008 - val_pearson_correlation: 0.0187 - val_pearson_correlation_log: 0.1952 - val_zero_penalty_metric: 0.1092 - learning_rate: 0.0010
Epoch 5/25
459/459 ━━━━━━━━━━━━━━━━━━━━ 204s 444ms/step - concordance_correlation_coefficient: 4.3632e-09 - cosine_similarity: 0.6136 - loss: 1.2862 - mean_absolute_error: 0.0357 - mean_squared_error: 39.5479 - pearson_correlation: 0.0418 - pearson_correlation_log: 0.3273 - zero_penalty_metric: 0.0979 - val_concordance_correlation_coefficient: 2.3310e-09 - val_cosine_similarity: 0.5145 - val_loss: 1.3289 

In [ ]:
trainer

In [ ]:
trainer.test()

In [47]:
crested.pp.change_regions_width(
    adata, 600
)  # change the adata width of the regions to 2114bp

In [48]:
#crested.pp.normalize_peaks(adata)

In [49]:
# %matplotlib inline
# crested.pl.bar.normalization_weights(adata, title="Normalization Weights per Cell Type")

In [50]:
# This one is a bit extreme, so either something should change with the inputs or just no normalisation

In [51]:
# Set the genome
genome = crested.Genome(
    "/home/bt392/rds/rds-bg200-hphi-gottgens/references/10x/refdata-cellranger-arc-mm10-2020-A-2.0.0/fasta/genome.fa", 
    "/home/bt392/rds/rds-bg200-hphi-gottgens/users/bt392/10_Eomes_invitro_gut/results/atac/chrombpnet/mm10.chrom.sizes"
)
crested.register_genome(
    genome
)  # Register the genome so that it can be used by the package

2025-03-21T14:13:21.050333+0000 INFO Genome genome registered.


In [52]:
adata.obs

,celltype
celltype,
Baso,Baso
BasoP,BasoP
cDC1,cDC1
cDC2a,cDC2a
cDC2b,cDC2b
cDC3,cDC3
Eos,Eos
Ery1,Ery1
Ery2,Ery2


In [54]:
adata = adata[:, [1,2]]

In [56]:
datamodule = crested.tl.data.AnnDataModule(
    adata,
    batch_size=256,  # lower this if you encounter OOM errors
    max_stochastic_shift=3,  # optional data augmentation to slightly reduce overfitting
    always_reverse_complement=True,  # default True. Will double the effective size of the training dataset.
)

In [57]:
crested.tl.zoo.chrombpnet

<function crested.tl.zoo._chrombpnet.chrombpnet(seq_len: int, num_classes: int, first_conv_filters: int = 512, first_conv_filter_size: int = 5, first_conv_pool_size: int = 0, first_conv_activation: str = 'gelu', first_conv_l2: float = 1e-05, first_conv_dropout: float = 0.1, n_dil_layers: int = 8, num_filters: int = 512, filter_size: int = 3, activation: str = 'relu', output_activation: str = 'softplus', l2: float = 1e-05, dropout: float = 0.1, batch_norm: bool = True, dense_bias: bool = True) -> keras.src.models.model.Model>

In [58]:
# Their default model is slightly different from the original bpnet
# mainly the width of the first convolutional filter is way smaller
model_architecture = crested.tl.zoo.simple_convnet(seq_len=600, num_classes=2)

In [59]:
# Load the default configuration for training a topic classication model
from crested.tl import default_configs, TaskConfig

config = default_configs("peak_regression")
print(config)

# If you want to change some small parameters to an existing config, you can do it like this
# For example, the default learning rate is 0.001, but you can change it to 0.0001
# config.optimizer.learning_rate = 0.0001

TaskConfig(optimizer=<keras.src.optimizers.adam.Adam object at 0x1507762a2890>, loss=<crested.tl.losses._cosinemse_log.CosineMSELogLoss object at 0x1507f24f1f50>, metrics=[<MeanAbsoluteError name=mean_absolute_error>, <MeanSquaredError name=mean_squared_error>, <CosineSimilarity name=cosine_similarity>, <PearsonCorrelation name=pearson_correlation>, <ConcordanceCorrelationCoefficient name=concordance_correlation_coefficient>, <PearsonCorrelationLog name=pearson_correlation_log>, <ZeroPenaltyMetric name=zero_penalty_metric>])


In [62]:
# setup the trainer
trainer = crested.tl.Crested(
    data=datamodule,
    model=model_architecture,
    config=config,
    project_name="/rds/project/rds-SDzz0CATGms/users/bt392/15_Kat_Atlas/results/crested",  # change to your liking
    run_name="trial4",  # change to your liking
    logger=None,  # or 'wandb', 'tensorboard'
)

In [63]:
# train the model
trainer.fit(epochs=25, early_stopping_patience=4)

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequence (InputLayer)           │ (None, 600, 4)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_12 (Conv1D)              │ (None, 588, 192)       │        10,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_16          │ (None, 588, 192)       │           768 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_16 (Activation)      │ (None, 588, 192)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_12 (MaxPooling1D) │ (None, 73, 192)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 73, 192)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_13 (Conv1D)              │ (None, 67, 256)        │       344,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_17          │ (None, 67, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_17 (Activation)      │ (None, 67, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_13 (MaxPooling1D) │ (None, 33, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 33, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_14 (Conv1D)              │ (None, 27, 512)        │       918,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_18          │ (None, 27, 512)        │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_18 (Activation)      │ (None, 27, 512)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_14 (MaxPooling1D) │ (None, 13, 512)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 13, 512)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 6656)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │     1,704,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_19          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_19 (Activation)      │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ denseblock_dense (Dense)        │ (None, 8)              │         2,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ denseblock_batchnorm            │ (None, 8)              │            3

 Total params: 2,983,674 (11.38 MB)

 Trainable params: 2,981,226 (11.37 MB)

 Non-trainable params: 2,448 (9.56 KB)

None
2025-03-21T14:14:17.574706+0000 INFO Loading sequences into memory...


100%|██████████| 2/2 [00:00<00:00, 13189.64it/s]


IndexError: list index out of range